# Producer B — Camera Event Stream (Camera 2)

This notebook implements the Kafka producer for **Camera 2** of the AWAS traffic monitoring pipeline. It reads `camera_event_B.csv`, groups records into batches by `batch_id`, and publishes each batch to the Kafka topic `camera-events-B` at a controlled rate to simulate a live roadside camera stream.

---

## Role in the Pipeline

Producer B simulates an independent roadside camera feed by continuously publishing Camera 2 events to Kafka. These streamed events are later consumed by Spark Structured Streaming for violation detection and cross-camera joins.

camera_event_B.csv → [Producer B] → Kafka topic: camera-events-B → Spark Structured Streaming



---

## EDA Findings

| Check | Finding |
|---|---|
| Camera IDs | Camera 2 only |
| Timestamp ordering | Not ordered within batches |
| Null speed readings | None found |
| Duplicate `car_plate` | 147 duplicates |
| Shared A/B plates | 9,844 matches |
| `batch_id` | Local to each producer |

These findings justified using Spark watermarks and time-window joins.

---

## Each Kafka message contains:


- `event_id` — unique camera event identifier
- `batch_id` — batch grouping key (local to Producer B)
- `car_plate` — vehicle registration plate (whitespace-stripped)
- `camera_id` — always `2` for this producer
- `timestamp` — original event time when the vehicle passed Camera 2
- `speed_reading` — recorded speed in km/h, rounded to 2 dp
- `producer_id` — always `'B'`
- `produced_at` — UTC ingestion timestamp added at publish time
- `batch_size` — total events in this batch

---

## Key Parameters

| Parameter | Value | Purpose |
|---|---|---|
| `BATCH_SLEEP_SECS` | `2` | Simulates streaming |
| `TOPIC` | `camera-events-B` | Kafka topic |
| Error handling | `try/except` | Skips malformed rows |

---

## batch_id Behaviour

`batch_id` groups events into publishing batches. Producer B has **294,206 unique batch IDs**. Since batch IDs are not shared across producers, Spark joins use `car_plate` and event-time windows instead.

In [1]:
# Use %pip to ensure the package is linked to this specific notebook kernel
%pip install kafka-python

Note: you may need to restart the kernel to use updated packages.


In [2]:
import csv
import json
import time
from datetime import datetime, timezone
from collections import defaultdict
from kafka import KafkaProducer         
from kafka.errors import KafkaError     

## Implementation Overview

The producer consists of three main functions:

1. **`load_batches(filepath)`** -reads the CSV, groups rows by `batch_id`, converts fields to correct types, and skips malformed rows safely.

2. **`connect_producer(broker)`** -creates and connects a JSON-based `KafkaProducer`.

3. **`publish_batches(producer, batches)`** -publishes one batch at a time, adds `produced_at` and `batch_size` metadata, flushes messages, logs checkpoints, and sleeps between batches to simulate streaming.

For demonstration purposes, the producer may be stopped after ~10–15 batches since full dataset completion is unnecessary during live demo execution.

In [3]:
#Configuration 
HOST_IP = '192.168.68.55'
KAFKA_BROKER = '192.168.0.175:9092'
TOPIC   = 'camera-events-B'
PRODUCER_ID    = 'B'
DATA_FILE  = '../data/camera_event_B.csv'
BATCH_SLEEP_SECS = 2    # seconds between batches

In [4]:
def load_batches(filepath):
    """
    Read camera event CSV and group rows by batch_id.

    Args:
        filepath (str): Path to the camera event CSV file.

    Returns:
        list: Sorted list of (batch_id, [events]) tuples ordered by batch_id.
    """
    # group all rows by batch_id so we can send them together
    batches = defaultdict(list)
    skipped = 0

    with open(filepath, newline='', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            try:
                event = {
                    'event_id': str(row['event_id']),
                    'batch_id': int(row['batch_id']),
                    'car_plate': row['car_plate'].strip(),
                    'camera_id': int(row['camera_id']),
                    'timestamp': row['timestamp'],
                    'speed_reading': round(float(row['speed_reading']), 2),
                    'producer_id': PRODUCER_ID
                }
                batches[event['batch_id']].append(event)
            except (KeyError, ValueError) as e:
                skipped += 1
                print(f'[WARN] Skipping malformed row: {e} | row={row}')

    total_events = sum(len(v) for v in batches.values())
    print(f'[INFO] Loaded {total_events} events across {len(batches)} batches. Skipped {skipped} malformed rows.')
    return sorted(batches.items())

In [5]:
def connect_producer(broker):
    """Connect to Kafka broker and return a JSON producer instance.

    Args:
        broker (str): Kafka broker address in host:port format.

    Returns:
        KafkaProducer: Connected producer ready to publish messages.
    """
    producer = KafkaProducer(
        bootstrap_servers=[broker],
        value_serializer=lambda v: json.dumps(v).encode('utf-8'),
        api_version=(2, 5, 0)  
    )
    # Ensure this print matches the producer (B or C)
    print(f'[INFO] Producer {PRODUCER_ID} connected → {broker} | topic: {TOPIC}')
    return producer

In [6]:
def publish_batches(producer, batches):
    """
    Publish all batches to the Kafka topic one batch at a time.

    Per batch: stamps produced_at and batch_size on each event,
    sends all events, flushes, then sleeps before next batch.
    Logs checkpoint (next batch_id) so restarts are possible after interruption.

    Args:
        producer (KafkaProducer): Conected Kafka producer.
        batches (list): Sorted list of (batch_id, [events]) tuples.
    """
    total_sent = 0

    for batch_id, events in batches:
        produced_at = datetime.now(timezone.utc).isoformat()
        batch_size  = len(events)
        batch_sent  = 0

        for event in events:
            event['produced_at'] = produced_at
            event['batch_size']  = batch_size
            try:
                producer.send(TOPIC, value=event)
                batch_sent += 1
            except KafkaError as e:
                print(f'[ERROR] Failed to send event {event["event_id"]}: {e}')

        producer.flush()
        total_sent += batch_sent
        print(f'[batch_id={batch_id:>6}] sent={batch_sent:>3} | '
              f'total={total_sent:>6} | next checkpoint: batch_id={batch_id + 1}')
        time.sleep(BATCH_SLEEP_SECS)

    print(f'[INFO] Producer B complete. Total sent: {total_sent}')

## Execution Notes

- Producers A, B, and C should be run **concurrently in separate Jupyter tabs** during the demo so Spark receives events from all three topics simultaneously.
- If your network changes (e.g. reconnecting to hotspot), update `HOST_IP` before re-running — the broker address is tied to your machine's current IPv4.
- If a `NoBrokersAvailable` error appears, confirm the Kafka container is running with `docker ps` and that the IP matches `ipconfig` output.

In [ ]:
# Main:
batches  = load_batches(DATA_FILE)
producer = connect_producer(KAFKA_BROKER)

try:
    publish_batches(producer, batches)
except KeyboardInterrupt:
    print('[INFO] Producer B interrupted by user.')
finally:
    producer.close()
    print('[INFO] Producer B connection closed.')

[INFO] Loaded 556340 events across 294206 batches. Skipped 0 malformed rows.
[INFO] Producer B connected → 192.168.0.175:9092 | topic: camera-events-B
[batch_id=     1] sent=  5 | total=     5 | next checkpoint: batch_id=2
[batch_id=     2] sent=  1 | total=     6 | next checkpoint: batch_id=3
[batch_id=     3] sent=  1 | total=     7 | next checkpoint: batch_id=4
[batch_id=     4] sent=  4 | total=    11 | next checkpoint: batch_id=5
[batch_id=     5] sent=  4 | total=    15 | next checkpoint: batch_id=6
[batch_id=     6] sent=  2 | total=    17 | next checkpoint: batch_id=7
[batch_id=     7] sent=  3 | total=    20 | next checkpoint: batch_id=8
[batch_id=     8] sent=  1 | total=    21 | next checkpoint: batch_id=9
[batch_id=     9] sent=  2 | total=    23 | next checkpoint: batch_id=10
[batch_id=    10] sent=  2 | total=    25 | next checkpoint: batch_id=11
[batch_id=    11] sent=  1 | total=    26 | next checkpoint: batch_id=12
[batch_id=    12] sent=  1 | total=    27 | next check

[batch_id=   112] sent=  1 | total=   213 | next checkpoint: batch_id=113
[batch_id=   113] sent=  2 | total=   215 | next checkpoint: batch_id=114
[batch_id=   114] sent=  5 | total=   220 | next checkpoint: batch_id=115
[batch_id=   115] sent=  2 | total=   222 | next checkpoint: batch_id=116
[batch_id=   116] sent=  3 | total=   225 | next checkpoint: batch_id=117
[batch_id=   117] sent=  2 | total=   227 | next checkpoint: batch_id=118
[batch_id=   118] sent=  1 | total=   228 | next checkpoint: batch_id=119
[batch_id=   119] sent=  1 | total=   229 | next checkpoint: batch_id=120
[batch_id=   120] sent=  5 | total=   234 | next checkpoint: batch_id=121
[batch_id=   121] sent=  1 | total=   235 | next checkpoint: batch_id=122
[batch_id=   122] sent=  3 | total=   238 | next checkpoint: batch_id=123
[batch_id=   123] sent=  2 | total=   240 | next checkpoint: batch_id=124
[batch_id=   124] sent=  1 | total=   241 | next checkpoint: batch_id=125
[batch_id=   125] sent=  2 | total=   

[batch_id=   223] sent=  5 | total=   435 | next checkpoint: batch_id=224
[batch_id=   224] sent=  2 | total=   437 | next checkpoint: batch_id=225
[batch_id=   225] sent=  1 | total=   438 | next checkpoint: batch_id=226
[batch_id=   226] sent=  4 | total=   442 | next checkpoint: batch_id=227
[batch_id=   227] sent=  3 | total=   445 | next checkpoint: batch_id=228
[batch_id=   228] sent=  1 | total=   446 | next checkpoint: batch_id=229
[batch_id=   229] sent=  1 | total=   447 | next checkpoint: batch_id=230
[batch_id=   230] sent=  1 | total=   448 | next checkpoint: batch_id=231
[batch_id=   231] sent=  4 | total=   452 | next checkpoint: batch_id=232
[batch_id=   232] sent=  3 | total=   455 | next checkpoint: batch_id=233
[batch_id=   233] sent=  1 | total=   456 | next checkpoint: batch_id=234
[batch_id=   234] sent=  5 | total=   461 | next checkpoint: batch_id=235
[batch_id=   235] sent=  2 | total=   463 | next checkpoint: batch_id=236
[batch_id=   236] sent=  2 | total=   

[batch_id=   334] sent=  2 | total=   638 | next checkpoint: batch_id=335
[batch_id=   335] sent=  3 | total=   641 | next checkpoint: batch_id=336
[batch_id=   336] sent=  1 | total=   642 | next checkpoint: batch_id=337
[batch_id=   337] sent=  1 | total=   643 | next checkpoint: batch_id=338
[batch_id=   338] sent=  4 | total=   647 | next checkpoint: batch_id=339
[batch_id=   339] sent=  2 | total=   649 | next checkpoint: batch_id=340
[batch_id=   340] sent=  2 | total=   651 | next checkpoint: batch_id=341
[batch_id=   341] sent=  1 | total=   652 | next checkpoint: batch_id=342
[batch_id=   342] sent=  1 | total=   653 | next checkpoint: batch_id=343
[batch_id=   343] sent=  1 | total=   654 | next checkpoint: batch_id=344
[batch_id=   344] sent=  1 | total=   655 | next checkpoint: batch_id=345
[batch_id=   345] sent=  2 | total=   657 | next checkpoint: batch_id=346
[batch_id=   346] sent=  1 | total=   658 | next checkpoint: batch_id=347
[batch_id=   347] sent=  2 | total=   

[batch_id=   445] sent=  1 | total=   836 | next checkpoint: batch_id=446
[batch_id=   446] sent=  2 | total=   838 | next checkpoint: batch_id=447
[batch_id=   447] sent=  1 | total=   839 | next checkpoint: batch_id=448
[batch_id=   448] sent=  1 | total=   840 | next checkpoint: batch_id=449
[batch_id=   449] sent=  1 | total=   841 | next checkpoint: batch_id=450
[batch_id=   450] sent=  3 | total=   844 | next checkpoint: batch_id=451
[batch_id=   451] sent=  1 | total=   845 | next checkpoint: batch_id=452
[batch_id=   452] sent=  1 | total=   846 | next checkpoint: batch_id=453
[batch_id=   453] sent=  3 | total=   849 | next checkpoint: batch_id=454
[batch_id=   454] sent=  3 | total=   852 | next checkpoint: batch_id=455
[batch_id=   455] sent=  1 | total=   853 | next checkpoint: batch_id=456
[batch_id=   456] sent=  1 | total=   854 | next checkpoint: batch_id=457
[batch_id=   457] sent=  3 | total=   857 | next checkpoint: batch_id=458
[batch_id=   458] sent=  2 | total=   

...

[batch_id=  3640] sent=  1 | total=  6822 | next checkpoint: batch_id=3641
[batch_id=  3641] sent=  1 | total=  6823 | next checkpoint: batch_id=3642
[batch_id=  3642] sent=  3 | total=  6826 | next checkpoint: batch_id=3643
[batch_id=  3643] sent=  1 | total=  6827 | next checkpoint: batch_id=3644
[batch_id=  3644] sent=  4 | total=  6831 | next checkpoint: batch_id=3645
[batch_id=  3645] sent=  1 | total=  6832 | next checkpoint: batch_id=3646
[batch_id=  3646] sent=  2 | total=  6834 | next checkpoint: batch_id=3647
[batch_id=  3647] sent=  4 | total=  6838 | next checkpoint: batch_id=3648
[batch_id=  3648] sent=  1 | total=  6839 | next checkpoint: batch_id=3649
[batch_id=  3649] sent=  3 | total=  6842 | next checkpoint: batch_id=3650
[batch_id=  3650] sent=  2 | total=  6844 | next checkpoint: batch_id=3651
[batch_id=  3651] sent=  1 | total=  6845 | next checkpoint: batch_id=3652
[batch_id=  3652] sent=  1 | total=  6846 | next checkpoint: batch_id=3653
[batch_id=  3653] sent=  

[batch_id=  3750] sent=  2 | total=  7031 | next checkpoint: batch_id=3751
[batch_id=  3751] sent=  4 | total=  7035 | next checkpoint: batch_id=3752
[batch_id=  3752] sent=  1 | total=  7036 | next checkpoint: batch_id=3753
[batch_id=  3753] sent=  3 | total=  7039 | next checkpoint: batch_id=3754
[batch_id=  3754] sent=  1 | total=  7040 | next checkpoint: batch_id=3755
[batch_id=  3755] sent=  1 | total=  7041 | next checkpoint: batch_id=3756
[batch_id=  3756] sent=  2 | total=  7043 | next checkpoint: batch_id=3757
[batch_id=  3757] sent=  1 | total=  7044 | next checkpoint: batch_id=3758
[batch_id=  3758] sent=  1 | total=  7045 | next checkpoint: batch_id=3759
[batch_id=  3759] sent=  2 | total=  7047 | next checkpoint: batch_id=3760
[batch_id=  3760] sent=  1 | total=  7048 | next checkpoint: batch_id=3761
[batch_id=  3761] sent=  2 | total=  7050 | next checkpoint: batch_id=3762
[batch_id=  3762] sent=  2 | total=  7052 | next checkpoint: batch_id=3763
[batch_id=  3763] sent=  

[batch_id=  3860] sent=  2 | total=  7241 | next checkpoint: batch_id=3861
[batch_id=  3861] sent=  1 | total=  7242 | next checkpoint: batch_id=3862
[batch_id=  3862] sent=  2 | total=  7244 | next checkpoint: batch_id=3863
[batch_id=  3863] sent=  5 | total=  7249 | next checkpoint: batch_id=3864
[batch_id=  3864] sent=  3 | total=  7252 | next checkpoint: batch_id=3865
[batch_id=  3865] sent=  2 | total=  7254 | next checkpoint: batch_id=3866
[batch_id=  3866] sent=  3 | total=  7257 | next checkpoint: batch_id=3867
[batch_id=  3867] sent=  1 | total=  7258 | next checkpoint: batch_id=3868
[batch_id=  3868] sent=  2 | total=  7260 | next checkpoint: batch_id=3869
[batch_id=  3869] sent=  1 | total=  7261 | next checkpoint: batch_id=3870
[batch_id=  3870] sent=  4 | total=  7265 | next checkpoint: batch_id=3871
[batch_id=  3871] sent=  1 | total=  7266 | next checkpoint: batch_id=3872
[batch_id=  3872] sent=  1 | total=  7267 | next checkpoint: batch_id=3873
[batch_id=  3873] sent=  

[batch_id=  3970] sent=  2 | total=  7465 | next checkpoint: batch_id=3971
[batch_id=  3971] sent=  1 | total=  7466 | next checkpoint: batch_id=3972
[batch_id=  3972] sent=  2 | total=  7468 | next checkpoint: batch_id=3973
[batch_id=  3973] sent=  1 | total=  7469 | next checkpoint: batch_id=3974
[batch_id=  3974] sent=  1 | total=  7470 | next checkpoint: batch_id=3975
[batch_id=  3975] sent=  2 | total=  7472 | next checkpoint: batch_id=3976
[batch_id=  3976] sent=  1 | total=  7473 | next checkpoint: batch_id=3977
[batch_id=  3977] sent=  2 | total=  7475 | next checkpoint: batch_id=3978
[batch_id=  3978] sent=  2 | total=  7477 | next checkpoint: batch_id=3979
[batch_id=  3979] sent=  1 | total=  7478 | next checkpoint: batch_id=3980
[batch_id=  3980] sent=  2 | total=  7480 | next checkpoint: batch_id=3981
[batch_id=  3981] sent=  1 | total=  7481 | next checkpoint: batch_id=3982
[batch_id=  3982] sent=  2 | total=  7483 | next checkpoint: batch_id=3983
[batch_id=  3983] sent=  

[batch_id=  4080] sent=  1 | total=  7665 | next checkpoint: batch_id=4081
[batch_id=  4081] sent=  2 | total=  7667 | next checkpoint: batch_id=4082
[batch_id=  4082] sent=  1 | total=  7668 | next checkpoint: batch_id=4083
[batch_id=  4083] sent=  2 | total=  7670 | next checkpoint: batch_id=4084
[batch_id=  4084] sent=  1 | total=  7671 | next checkpoint: batch_id=4085
[batch_id=  4085] sent=  1 | total=  7672 | next checkpoint: batch_id=4086
[batch_id=  4086] sent=  4 | total=  7676 | next checkpoint: batch_id=4087
[batch_id=  4087] sent=  2 | total=  7678 | next checkpoint: batch_id=4088
[batch_id=  4088] sent=  1 | total=  7679 | next checkpoint: batch_id=4089
[batch_id=  4089] sent=  3 | total=  7682 | next checkpoint: batch_id=4090
[batch_id=  4090] sent=  6 | total=  7688 | next checkpoint: batch_id=4091
[batch_id=  4091] sent=  2 | total=  7690 | next checkpoint: batch_id=4092
[batch_id=  4092] sent=  3 | total=  7693 | next checkpoint: batch_id=4093
[batch_id=  4093] sent=  